# Diário Oficial – Monitoramento de Convocações

Notebook simples para testar:
1. Upload do PDF do Diário
2. Extração de convocados
3. Join com nomes monitorados (definidos em YAML no código)
4. Print dos resultados

Funciona diretamente no Google Colab.

In [ ]:
!pip install pdfplumber pandas pyyaml

## Upload do PDF do Diário Oficial

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print('Arquivo carregado:', pdf_path)

## Definir usuários monitorados (YAML no código)

In [ ]:
import yaml

yaml_users = """
usuarios:
  - id: 1
    nome: MICHELE SILVA
    email: michele@email.com

  - id: 2
    nome: JOAO PEREIRA
    email: joao@email.com
"""

usuarios = yaml.safe_load(yaml_users)["usuarios"]

print("Usuarios monitorados:")
for u in usuarios:
    print(u)

## Extrair texto do PDF

In [ ]:
import pdfplumber

texto = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        t = page.extract_text()
        if t:
            texto += t + "\n"

print("Total de caracteres extraídos:", len(texto))
print("Preview:\n")
print(texto[:2000])

## Cortar seção ATOS OFICIAIS

In [ ]:
if "ATOS OFICIAIS" in texto:
    texto = texto.split("ATOS OFICIAIS",1)[1]

print(texto[:2000])

## Encontrar editais de convocação

In [ ]:
import re

pattern = r"(EDITAL DE CONVOCAÇÃO.*?)(?=EDITAL DE CONVOCAÇÃO|$)"

editais = re.findall(pattern, texto, re.DOTALL | re.IGNORECASE)

print("Total de editais encontrados:", len(editais))

if editais:
    print("Preview do primeiro edital:\n")
    print(editais[0][:1000])

## Extrair possíveis convocados

In [ ]:
convocados = []

pattern = r"(\d+)\s+([A-ZÁÉÍÓÚÃÕÇ ]{5,})"

for edital in editais:
    matches = re.findall(pattern, edital)
    for m in matches:
        convocados.append({
            "classificacao": m[0],
            "nome": m[1].strip()
        })

print("Total convocados encontrados:", len(convocados))

for c in convocados[:10]:
    print(c)

## Fazer join com monitorados

In [ ]:
matches = []

for c in convocados:
    for u in usuarios:
        if c["nome"].strip().upper() == u["nome"].strip().upper():
            matches.append({
                "usuario_id": u["id"],
                "usuario_nome": u["nome"],
                "classificacao": c["classificacao"]
            })

print("\nRESULTADOS ENCONTRADOS")
print("======================")

if not matches:
    print("Nenhum monitorado encontrado.")

for m in matches:
    print(m)